In [5]:
import re
import string
from typing import List, Dict, Tuple, Any, Set
import json
import copy

class PodcastIRDatasetProcessor:
    def __init__(self, chunk_size=100, overlap=50):
        """
        Initialize the processor with segmentation parameters.
        
        Args:
            chunk_size: Number of words per segment
            overlap: Number of words of overlap between segments
        """
        self.chunk_size = chunk_size
        self.overlap = overlap
    
    def normalize_text(self, text: str) -> str:
        """
        Normalize text by removing punctuation and converting to lowercase
        for matching purposes.
        """
        # Remove punctuation and convert to lowercase
        translator = str.maketrans('', '', string.punctuation)
        return text.translate(translator).lower()
    
    def find_question_spans(self, podcast_id: str, transcript: str, questions: List[str]) -> List[Dict[str, Any]]:
        """
        Find spans in the transcript that contain the questions from QA pairs.
        
        Args:
            podcast_id: Identifier for the podcast
            transcript: The full podcast transcript
            questions: List of questions from QA pairs
            
        Returns:
            List of dictionaries with question spans and metadata
        """
        # Normalize the transcript for matching
        normalized_transcript = self.normalize_text(transcript)
        
        # Store original transcript for returning original text
        transcript_words = transcript.split()
        normalized_transcript_words = normalized_transcript.split()
        
        question_spans = []
        
        for q_idx, question in enumerate(questions):
            normalized_question = self.normalize_text(question)
            normalized_question_words = normalized_question.split()
            question_length = len(normalized_question_words)
            
            # Sliding window search through transcript
            for i in range(len(normalized_transcript_words) - question_length + 1):
                window = ' '.join(normalized_transcript_words[i:i+question_length])
                
                # If we found a match
                if normalized_question == window:
                    # Get original text with original case and punctuation
                    original_span_text = ' '.join(transcript_words[i:i+question_length])
                    
                    # Store the span information
                    question_spans.append({
                        'question_id': q_idx,
                        'question_text': question,
                        'span_text': original_span_text,
                        'start_word_index': i,
                        'end_word_index': i + question_length,
                        'podcast_id': podcast_id
                    })
        
        return question_spans
    
    def find_answer_spans(self, podcast_id: str, transcript: str, answers: List[str]) -> List[Dict[str, Any]]:
        """
        Find spans in the transcript that contain the answers from QA pairs.
        Similar to question span finding but for answers.
        """
        # Implementation follows the same pattern as find_question_spans
        normalized_transcript = self.normalize_text(transcript)
        
        transcript_words = transcript.split()
        normalized_transcript_words = normalized_transcript.split()
        
        answer_spans = []
        
        for a_idx, answer in enumerate(answers):
            normalized_answer = self.normalize_text(answer)
            normalized_answer_words = normalized_answer.split()
            answer_length = len(normalized_answer_words)
            
            for i in range(len(normalized_transcript_words) - answer_length + 1):
                window = ' '.join(normalized_transcript_words[i:i+answer_length])
                
                if normalized_answer == window:
                    original_span_text = ' '.join(transcript_words[i:i+answer_length])
                    
                    answer_spans.append({
                        'answer_id': a_idx,
                        'answer_text': answer,
                        'span_text': original_span_text,
                        'start_word_index': i,
                        'end_word_index': i + answer_length,
                        'podcast_id': podcast_id
                    })
        
        return answer_spans
    
    def segment_transcript(self, podcast_id: str, transcript: str, question_spans: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """
        Segment the transcript into overlapping chunks and apply information hiding.
        
        Args:
            podcast_id: Identifier for the podcast
            transcript: The full podcast transcript
            question_spans: Question spans to hide
            
        Returns:
            List of segmented documents with questions removed
        """
        # Split transcript into words
        words = transcript.split()
        total_words = len(words)
        
        # Create a set of word indices to hide (questions)
        words_to_hide = set()
        for span in question_spans:
            for i in range(span['start_word_index'], span['end_word_index']):
                words_to_hide.add(i)
        
        # Create segments with overlap
        segments = []
        for start_idx in range(0, total_words, self.chunk_size - self.overlap):
            end_idx = min(start_idx + self.chunk_size, total_words)
            segment_id = f"{podcast_id}_seg{len(segments)+1:03d}"
            
            # Apply information hiding by removing question words
            segment_words = []
            hidden_indices = []
            
            for i in range(start_idx, end_idx):
                if i in words_to_hide:
                    # Replace with [QUESTION_REMOVED] as a marker
                    segment_words.append("[QUESTION_REMOVED]")
                    hidden_indices.append(i - start_idx)  # Relative to segment start
                else:
                    segment_words.append(words[i])
            
            # Store segment information
            segments.append({
                'segment_id': segment_id,
                'podcast_id': podcast_id,
                'start_idx': start_idx,
                'end_idx': end_idx,
                'original_text': ' '.join(words[start_idx:end_idx]),
                'processed_text': ' '.join(segment_words),
                'hidden_indices': hidden_indices,
                'word_count': end_idx - start_idx
            })
            
            # If we've reached the end of the transcript, break
            if end_idx == total_words:
                break
        
        return segments
    
    def create_qrels(self, segments: List[Dict[str, Any]], answer_spans: List[Dict[str, Any]], qa_pairs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """
        Create query-document relevance judgments (qrels).
        
        Args:
            segments: List of transcript segments
            answer_spans: List of answer spans
            qa_pairs: List of QA pairs
            
        Returns:
            List of relevance judgments
        """
        qrels = []
        
        # Create a mapping from answer_id to query_id
        answer_to_query = {}
        for qa in qa_pairs:
            answer_to_query[qa['answer_id']] = qa['query_id']
        
        # For each segment, check if it contains any answer spans
        for segment in segments:
            segment_start = segment['start_idx']
            segment_end = segment['end_idx']
            
            for answer_span in answer_spans:
                answer_start = answer_span['start_word_index']
                answer_end = answer_span['end_word_index']
                
                # Check if there's overlap between the segment and answer span
                if (answer_start < segment_end and answer_end > segment_start and 
                    answer_span['podcast_id'] == segment['podcast_id']):
                    
                    # Calculate overlap percentage
                    overlap_start = max(segment_start, answer_start)
                    overlap_end = min(segment_end, answer_end)
                    overlap_length = overlap_end - overlap_start
                    answer_length = answer_end - answer_start
                    
                    # Calculate relevance grade based on how much of the answer is contained
                    if overlap_length == answer_length:
                        relevance = 3  # Complete answer
                    elif overlap_length > answer_length * 0.5:
                        relevance = 2  # Major part of the answer
                    else:
                        relevance = 1  # Minor part of the answer
                    
                    query_id = answer_to_query.get(answer_span['answer_id'])
                    if query_id:
                        qrels.append({
                            'query_id': query_id,
                            'segment_id': segment['segment_id'],
                            'relevance': relevance,
                            'answer_id': answer_span['answer_id']
                        })
        
        return qrels
    
    def process_podcasts(self, podcasts: Dict[str, str], qa_pairs: List[Dict[str, Any]]) -> Dict[str, Any]:
        """
        Process all podcasts to create the IR dataset.
        
        Args:
            podcasts: Dictionary mapping podcast IDs to transcripts
            qa_pairs: List of QA pairs with question, answer, and podcast_id fields
            
        Returns:
            Dictionary with queries, documents, and qrels
        """
        # Group QA pairs by podcast
        podcast_qa = {}
        for i, qa in enumerate(qa_pairs):
            podcast_id = qa['podcast_id']
            # Assign unique IDs if not present
            if 'query_id' not in qa:
                qa['query_id'] = f"q{i+1:03d}"
            if 'answer_id' not in qa:
                qa['answer_id'] = i
                
            if podcast_id not in podcast_qa:
                podcast_qa[podcast_id] = {'questions': [], 'answers': [], 'qa_pairs': []}
            podcast_qa[podcast_id]['questions'].append(qa['question'])
            podcast_qa[podcast_id]['answers'].append(qa['answer'])
            podcast_qa[podcast_id]['qa_pairs'].append(qa)
        
        # Process each podcast
        all_segments = []
        all_question_spans = []
        all_answer_spans = []
        
        for podcast_id, transcript in podcasts.items():
            if podcast_id in podcast_qa:
                # Find question and answer spans
                questions = podcast_qa[podcast_id]['questions']
                answers = podcast_qa[podcast_id]['answers']
                
                question_spans = self.find_question_spans(podcast_id, transcript, questions)
                answer_spans = self.find_answer_spans(podcast_id, transcript, answers)
                
                # Segment with information hiding
                segments = self.segment_transcript(podcast_id, transcript, question_spans)
                
                # Store results
                all_segments.extend(segments)
                all_question_spans.extend(question_spans)
                all_answer_spans.extend(answer_spans)
        
        # Create qrels
        all_qrels = self.create_qrels(all_segments, all_answer_spans, qa_pairs)
        
        # Format queries
        queries = [{'query_id': qa['query_id'], 'query_text': qa['question']} for qa in qa_pairs]
        
        # Return the complete dataset
        return {
            'queries': queries,
            'documents': all_segments,
            'qrels': all_qrels,
            'question_spans': all_question_spans,
            'answer_spans': all_answer_spans
        }
    
    def write_dataset_files(self, dataset: Dict[str, Any], output_dir: str) -> None:
        """
        Write the dataset to files in standard IR format.
        
        Args:
            dataset: The processed dataset
            output_dir: Directory to write the files to
        """
        import os
        import json
        
        os.makedirs(output_dir, exist_ok=True)
        
        # Write queries
        with open(f"{output_dir}/queries.json", 'w') as f:
            json.dump(dataset['queries'], f, indent=2)
        
        # Write documents
        with open(f"{output_dir}/docs.json", 'w') as f:
            json.dump(dataset['documents'], f, indent=2)
        
        # Write qrels in TREC format
        with open(f"{output_dir}/qrels.txt", 'w') as f:
            for qrel in dataset['qrels']:
                f.write(f"{qrel['query_id']} 0 {qrel['segment_id']} {qrel['relevance']}\n")
        
        # Write metadata
        with open(f"{output_dir}/metadata.json", 'w') as f:
            metadata = {
                'question_spans': dataset['question_spans'],
                'answer_spans': dataset['answer_spans'],
                'processing_info': {
                    'chunk_size': self.chunk_size,
                    'overlap': self.overlap,
                    'num_documents': len(dataset['documents']),
                    'num_queries': len(dataset['queries']),
                    'num_qrels': len(dataset['qrels'])
                }
            }
            json.dump(metadata, f, indent=2)

# Example usage
def example():
    # Sample data
    podcasts = {
        "pod1": "Welcome to our show. Today we'll discuss artificial intelligence in detail. What are the biggest challenges in AI today? Well, experts suggest several issues including algorithmic bias and energy consumption. Another concern is privacy implications of large language models. Let's explore these topics in more detail now.",
        "pod2": "In this episode about climate change: How will rising temperatures affect agriculture? Scientists predict crop yields may decrease by up to 25% in some regions. This could lead to food insecurity in vulnerable populations. We need to develop resilient farming practices to address these challenges."
    }
    
    qa_pairs = [
        {
            "podcast_id": "pod1",
            "question": "What are the biggest challenges in AI today?",
            "answer": "experts suggest several issues including algorithmic bias and energy consumption"
        },
        {
            "podcast_id": "pod2",
            "question": "How will rising temperatures affect agriculture?",
            "answer": "Scientists predict crop yields may decrease by up to 25% in some regions"
        }
    ]
    
    # Process podcasts
    processor = PodcastIRDatasetProcessor(chunk_size=10, overlap=5)  # Small chunks for example
    dataset = processor.process_podcasts(podcasts, qa_pairs)
    
    # Print some results
    print("Queries:")
    for query in dataset['queries']:
        print(f"  {query['query_id']}: {query['query_text']}")
    
    print("\nDocuments (showing first 2):")
    for doc in dataset['documents'][:2]:
        print(f"  {doc['segment_id']}: {doc['processed_text'][:50]}...")
    
    print("\nQRels:")
    for qrel in dataset['qrels']:
        print(f"  {qrel['query_id']} -> {qrel['segment_id']} (relevance: {qrel['relevance']})")

In [6]:
example()


Queries:
  q001: What are the biggest challenges in AI today?
  q002: How will rising temperatures affect agriculture?

Documents (showing first 2):
  pod1_seg001: Welcome to our show. Today we'll discuss artificia...
  pod1_seg002: we'll discuss artificial intelligence in detail. [...

QRels:
  q001 -> pod1_seg004 (relevance: 1)
  q001 -> pod1_seg005 (relevance: 3)
  q001 -> pod1_seg006 (relevance: 1)
  q001 -> pod2_seg002 (relevance: 1)
  q001 -> pod2_seg003 (relevance: 2)
  q001 -> pod2_seg004 (relevance: 2)
  q001 -> pod2_seg005 (relevance: 1)


-------------------------------

In [10]:
from pathlib import Path
import json
import pandas as pd
from typing import List

results_dir = "output/q_extraction/gpt-4o-mini-2024-07-18/q_extraction_v0.1/initial_ids_v0.1"
id_field = "id"
dataset = "output/data/spotify_candidates_v0.1.tsv"
ids_keep = "output/lists/initial_ids_v0.1.lst"

def remove_ids_in_df(df: pd.DataFrame, ids: set, id_col: str = "id") -> pd.DataFrame:
    """Filter DataFrame by IDs, keeping the order in the list."""
    df_filtered = df[~df[id_col].isin(ids)].copy()
    return df_filtered

def read_ids(file: str) -> List[str]:
    with open(file) as f:
        ids = [str(line.strip()) for line in f]
    return ids

def keep_ids_in_df(df: pd.DataFrame, ids: List, id_col: str = "id") -> pd.DataFrame:
    """Filter DataFrame by IDs, keeping the order in the list."""
    # Filter data by IDs:
    df_filtered = df[df[id_col].isin(ids)].copy()
    # Reorder according to IDs list:
    id_to_pos = {id_: pos for pos, id_ in enumerate(ids)}
    df_filtered['sort_pos'] = df_filtered[id_col].map(id_to_pos)
    df_filtered = df_filtered.sort_values('sort_pos').drop('sort_pos', axis=1)
    df_filtered = df_filtered.reset_index(drop=True).copy()
    return df_filtered

df = pd.read_csv(dataset, sep="\t")
df[id_field] = df[id_field].astype(str)

ids_keep = read_ids(ids_keep)
df = keep_ids_in_df(df, ids_keep, id_field)

print(f"Number of rows in the dataset: {len(df)}")

Number of rows in the dataset: 7000


In [11]:
# done_ids

In [12]:
results_files = sorted(Path(results_dir).glob("batch_*.jsonl"))
last_batch = 0
if results_files:
    done_ids = set()
    # grab all ids from all files of each json:
    for file in results_files:
        with open(file, "r") as f:
            for line in f:
                id_ = json.loads(line)["custom_id"]
                done_ids.add(id_)
    last_file = results_files[-1]
    last_batch = int(last_file.stem.split("_")[-1])
    # remove encoded_ids from dataset:
    if done_ids:
        print(f"Removing {len(done_ids)} already done documents")
        df = remove_ids_in_df(df, done_ids, id_field)
    else:
        print(f"No encoded IDs found in {results_dir}")

print(f"Number of rows left in the dataset: {len(df)}")

Removing 200 already done documents
Number of rows left in the dataset: 6800


------------------

In [2]:
import re
def extract_verdicts_with_bytes(text):
    properties = [
        "Information-seeking",
        "Self-contained"
    ]
    results = {}
    for prop in properties:
        pattern = rf"{prop}:.*?Verdict:\s*(yes|no)"
        match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
        if match:
            verdict = match.group(1).lower()
            verdict_start = match.start(1)
            verdict_end = match.end(1)
            byte_start = len(text[:verdict_start].encode('utf-8'))
            byte_end = len(text[:verdict_end].encode('utf-8'))
            bytes_idx = list(range(byte_start, byte_end))
            key = prop.lower().replace('-', '_')
            results[key] = {
                'verdict': verdict,
                'bytes_idx': bytes_idx
            }
        else:
            key = prop.lower().replace('-', '_')
            results[key] = None
    return results

y = "Information-seeking: XYZ. Verdict: yes.  \nSelf-contained: AA. Verdict: no."

print(extract_verdicts_with_bytes(y))

{'information_seeking': {'verdict': 'yes', 'bytes_idx': [35, 36, 37]}, 'self_contained': {'verdict': 'no', 'bytes_idx': [71, 72]}}


In [3]:
# Read first line of:
# output/q_quality/gpt-4o-mini-2024-07-18/debug_sample_v1/batch_00001.jsonl

import json

with open('output/q_quality/gpt-4o-mini-2024-07-18/debug_sample_v1/batch_00001.jsonl', 'r') as f:
    line = f.readline()
    data = json.loads(line)


In [4]:
logprobs = data["response"]["body"]["choices"][0]["logprobs"]["content"]

In [35]:
x = "Information-seeking: The question asks for information about the actions of a specific duo of hounds, which implies a general inquiry about their behavior or activities. However, it lacks context about which hounds are being referred to, making it somewhat ambiguous. Verdict: no.  \nSelf-contained: The question does not provide enough context or detail to be understood independently by someone not involved in the conversation. It does not specify which hounds are being discussed or what the expected actions might be. Verdict: no."

parsed_verdicts = extract_verdicts_with_bytes(x)
print(parsed_verdicts)

{'information_seeking': {'verdict': 'no', 'bytes_idx': [277, 278]}, 'self_contained': {'verdict': 'no', 'bytes_idx': [531, 532]}}


In [36]:
# Iterate over the logprobs and save the ones corresponding to the verdicts
def get_verdict_logprobs(logprobs, parsed_verdicts):
    byte_idx = 0
    res_dict = {}
    for token_dict in logprobs:
        token_bytes = token_dict["bytes"]
        n_bytes = len(token_bytes)
        byte_idx_end = byte_idx + n_bytes
        token_bytes_idx = list(range(byte_idx, byte_idx_end))
        # print(token_bytes_idx)
        for key, verdict_dict in parsed_verdicts.items():
            if verdict_dict is not None:
                if set(token_bytes_idx).intersection(verdict_dict["bytes_idx"]):
                    # print(key, token_dict['logprob'])
                    res_dict[key] = token_dict['logprob']
        byte_idx = byte_idx_end
    return res_dict

logprobs_dict = get_verdict_logprobs(logprobs, parsed_verdicts)
print(logprobs_dict)

{'information_seeking': -0.017984869, 'self_contained': 0.0}


In [37]:
parsed_verdicts

{'information_seeking': {'verdict': 'no', 'bytes_idx': [277, 278]},
 'self_contained': {'verdict': 'no', 'bytes_idx': [531, 532]}}

In [ ]:
# dicts with property: [verdict, logprob]
res = {k: [v["verdict"], logprobs_dict[k]] for k, v in parsed_verdicts.items() if v is not None}
res


{'information_seeking': ['no', -0.017984869], 'self_contained': ['no', 0.0]}

In [41]:
tuple(res["information_seeking"] + res["self_contained"])

('no', -0.017984869, 'no', 0.0)

In [7]:
# Read first line of:
# "output/q_extraction/gpt-4o-mini-2024-07-18/sample_ids_v2/self_contained_q_v3/batch_00001.jsonl"
import json

file = "output/q_extraction/gpt-4o-mini-2024-07-18/sample_ids_v2/self_contained_q_v3/batch_00001.jsonl"
with open(file, 'r') as f:
    line = f.readline()
    data = json.loads(line)

response = data["response"]["body"]["choices"][0]["message"]


In [9]:
print(response["content"])

Reasoning: The question must seek general factual knowledge that stands alone without needing extra context. The transcript focuses on the Michigan football defense and its players, but it does not present any clear, self-contained questions that fit the criteria. There are discussions about various players, their statistics, and predictions, but no direct inquiries that ask for information in a general sense. 

Question: N/A


In [11]:
# decode all bytes:
decoded = bytes(all_bytes).decode('utf-8')
print(decoded)

assert decoded == x

Information-seeking: The question asks for information about the actions of a specific duo of hounds, which implies a general inquiry about their behavior or activities. However, it lacks context about which hounds are being referred to, making it somewhat ambiguous. Verdict: no.  
Self-contained: The question does not provide enough context or detail to be understood independently by someone not involved in the conversation. It does not specify which hounds are being discussed or what the expected actions might be. Verdict: no.


In [7]:
x_bytes = list(x.encode('utf-8'))

In [8]:
bytes(x_bytes[277:278+1]).decode('utf-8')

'no'

In [1]:
bytes

bytes

In [3]:
# Read OLD/podcasts_qa_test.json:
import json

with open('OLD/podcasts_qa_test.json', 'r') as f:
    data = json.load(f)

# keep the 1st element of each list:
data = [x[0] for x in data]
data = [x for x in data if x != 'N/A']

print(*data, sep='\n')

What would you say is the biggest thing to focus on for a photo shoot?
What did you think about the conversation that took place when Tasha Skyped in?
What is the weirdest wave that you have ever surfed?
What do you think about the midcourt for the Australian team in the Netball World Cup?
How do you overcome it when you kind of sink into like oh my god everybody's gonna die and leave me like when that happened does it happen to you?
What are some simple steps to incorporate diet and fitness into your daily life?
What do I wish I would have known when I started this job?
How would you describe the heart of premier?
So if I need to lower my body fat percentage am I going to see in my abs?
What was it her older sister was considered?
What if you stop putting the pressure on yourself to love every single thing you do and instead learn to sit with that discomfort?
Do you think that because the gay marriage debate is so fresh in the mind of the Australian public that that is why it's had su

In [1]:
# Read a toy JSON with null values:
import json

x = '{"a": null, "b": "hello"}'
data = json.loads(x)
print(data)

{'a': None, 'b': 'hello'}


In [2]:
# Count tokens in dataset:
import tiktoken
import pandas as pd

def num_tokens_from_string(string: str, model_name: str) -> int:
    '''Returns the number of tokens in a text string.'''
    encoding = tiktoken.encoding_for_model(model_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

# prompt: text from "prompts/self_contained_qas_v3.prompt"
# inputs: from "output/data/spotify_candidates.tsv"

with open('prompts/self_contained_qas_v3.prompt', 'r') as f:
    prompt = f.read()

df = pd.read_csv('output/data/spotify_candidates.tsv', sep='\t')

input_tokens = df['text'].apply(lambda x: num_tokens_from_string(x, 'gpt-4o-mini-2024-07-18'))
prompt_tokens = num_tokens_from_string(prompt, 'gpt-4o-mini-2024-07-18')

In [4]:
mini_input_price = 0.075 / 1e6
mini_output_price = 0.3 / 1e6
regular_input_price = 1.25 / 1e6
regular_output_price = 5 / 1e6

n_tokens_input = input_tokens.sum() + prompt_tokens * len(df)
n_tokens_output = 1000 * len(df) # assuming 1000 tokens per output

mini_input_price = n_tokens_input * mini_input_price
mini_output_price = n_tokens_output * mini_output_price
regular_input_price = n_tokens_input * regular_input_price
regular_output_price = n_tokens_output * regular_output_price

print(f"Mini input price: {mini_input_price:.2f}")
print(f"Mini output price: {mini_output_price:.2f}")
print(f"Regular input price: {regular_input_price:.2f}")
print(f"Regular output price: {regular_output_price:.2f}")


Mini input price: 46.14
Mini output price: 26.53
Regular input price: 769.08
Regular output price: 442.12
